# MADA - Fase 2: Experimento Empírico

**Metodologia de Identificação de Viés Amplificado**

Este notebook contempla a fase do estudo que abrange a análise de anotações reais para detectar a amplificação de viés proposta pela metodologia.


### Conferencia do ambiente (Local x Colab)

In [1]:
import os
import sys

# 1. Detectar se o ambiente é o Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ambiente detectado: Google Colab")
    REPO_URL = "https://github.com/alexfeitosa72/MIVA.git"
    REPO_NAME = "MIVA"

    # Clonar apenas se a pasta ainda não existir no ambiente virtual do Colab
    if not os.path.exists(REPO_NAME):
        print(f"📥 Clonando {REPO_NAME}...")
        !git clone {REPO_URL}
    
    # Entrar na pasta do repositório no Colab
    if os.getcwd().split('/')[-1] != REPO_NAME:
        os.chdir(REPO_NAME)
    
    print(f"Diretório de trabalho ajustado: {os.getcwd()}")
else:
    print(f"Ambiente detectado: Local (VS Code/Jupyter)")
    print(f"Mantendo diretório atual: {os.getcwd()}")

# 2. Listar arquivos de forma compatível (funciona em Windows, Mac, Linux e Colab)
print("\nArquivos na pasta raiz:")
print(os.listdir('.'))

Ambiente detectado: Local (VS Code/Jupyter)
Mantendo diretório atual: b:\TechLab\MIVA

Arquivos na pasta raiz:
['.claude', '.git', '.lh', '.vscode', 'data', 'fase1_tratamento_logs.ipynb', 'fase2_dados_empiricos.ipynb', 'fase3_dados_sinteticos.ipynb', 'legado', 'MADA.md', 'MADA2.md', 'resultados_MADA.ipynb', 'xtra_analise_qualitativa.ipynb', '_refs']


## 1. Setup e Configurações

In [2]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    cohen_kappa_score,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from sklearn.base import clone

# Análise estatística
from scipy.stats import chi2_contingency

# Visualização
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

# Configurações
warnings.filterwarnings('ignore')

# ============================================================================
# ESTILO ACADÊMICO NATURE (azulado)
# ============================================================================
plt.style.use('seaborn-v0_8-whitegrid')

NATURE_BLUE = '#0C4A6E'
NATURE_BLUE_MED = '#0284C7'
NATURE_BLUE_LIGHT = '#38BDF8'
NATURE_GRAY = '#64748B'
NATURE_ACCENT = '#1E3A5F'

NATURE_PALETTE = [NATURE_BLUE, NATURE_BLUE_MED, NATURE_BLUE_LIGHT, NATURE_GRAY]

CORES_MODELOS_NATURE = {
    'SVM': '#0C4A6E',
    'NB': '#0284C7',
    'RF': '#0EA5E9',
    'LR': '#38BDF8'
}

COR_NEGATIVA = '#DC2626'
COR_NEUTRA = '#6B7280'
COR_POSITIVA = '#0284C7'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.labelweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})

sns.set_palette(NATURE_PALETTE)
pd.options.display.float_format = '{:.4f}'.format

print("[OK] Bibliotecas importadas com sucesso.")
print("[OK] Estilo acadêmico Nature aplicado.")


[OK] Bibliotecas importadas com sucesso.
[OK] Estilo acadêmico Nature aplicado.


In [3]:
SEED = 42
N_FOLDS = 5
INPUT_FILE = "data/logs_processados/MQD-1222_majoritarias.csv"

DATA_DIR = Path("data")
RESULTS_DIR = DATA_DIR / "resultados_empiricos"
GRAFICOS_DIR = RESULTS_DIR / "graficos_empiricos"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

CLASSE_MAP = {'negativa': -1, 'neutra': 0, 'positiva': 1}
CLASSE_REVERSO = {v: k for k, v in CLASSE_MAP.items()}

print("=" * 80)
print("CONFIGURAÇÕES")
print("=" * 80)
print(f"Seed: {SEED}")
print(f"K-Fold CV: {N_FOLDS} folds")
print(f"Arquivo: {INPUT_FILE}")
print("=" * 80)


CONFIGURAÇÕES
Seed: 42
K-Fold CV: 5 folds
Arquivo: data/logs_processados/MQD-1222_majoritarias.csv


In [ ]:
# Modelos de classificação (hiperparâmetros padrão)
modelos = {
    'SVM': SVC(kernel='linear', C=1.0, random_state=SEED),
    'NB': MultinomialNB(alpha=1.0),
    'RF': RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'LR': LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, n_jobs=-1)
}

print("="*80)
print("MODELOS CONFIGURADOS")
print("="*80)
for nome, modelo in modelos.items():
    params = modelo.get_params()
    params_rel = {k: v for k, v in params.items() 
                  if k in ['C', 'alpha', 'kernel', 'n_estimators', 'solver', 'random_state']}
    print(f"\n{nome}:")
    for param, valor in params_rel.items():
        print(f"    {param}: {valor}")

## MÓDULO I: Análise de Consistência Preliminar (Baseline)

Estabelece a linha de base da concordância entre os dois grupos de anotadores.

In [ ]:
# Carregar dados
print("="*80)
print("MÓDULO I: BASELINE")
print("="*80)

df = pd.read_csv(INPUT_FILE, sep='\t')
print(f"\n[OK] {len(df)} frases carregadas.")
print(f"[OK] Colunas: {list(df.columns)}.")

df.head()

In [ ]:
# Mapear classes para valores numéricos
y_masculino = df['classificacao_majoritaria_masculino'].map(CLASSE_MAP)
y_feminino = df['classificacao_majoritaria_feminino'].map(CLASSE_MAP)

print("\nMapeamento:")
for classe, valor in CLASSE_MAP.items():
    print(f"  {classe:>10} -> {valor:>2}")

In [ ]:
# Calcular métricas de concordância inicial
kappa_inicial = cohen_kappa_score(y_masculino, y_feminino)
concordancia = np.mean(y_masculino == y_feminino)

contingency = pd.crosstab(y_masculino, y_feminino)
chi2, p_value, dof, expected = chi2_contingency(contingency)

n = len(y_masculino)
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f"\nMétricas de Baseline:")
print(f"  κ_inicial (Cohen)    : {kappa_inicial:.4f}")
print(f"  Concordância         : {concordancia:.4f} ({100*concordancia:.2f}%)")
print(f"  χ²                   : {chi2:.2f} (p < {p_value:.2e})")
print(f"  Cramér's V           : {cramers_v:.4f}")

df_baseline = pd.DataFrame({
    'metrica': ['Cohen_Kappa', 'Concordancia', 
                'Chi2', 'p_value', 'Cramers_V', 'N_amostras'],
    'valor': [kappa_inicial, concordancia, 
              chi2, p_value, cramers_v, n]
})

df_baseline

In [ ]:
# Bootstrap para IC do κ_inicial
print("\nCalculando intervalo de confiança (bootstrap)...")

n_bootstrap = 1000
kappas_bootstrap = []

for _ in range(n_bootstrap):
    idx = np.random.choice(len(y_masculino), size=len(y_masculino), replace=True)
    kappa_sample = cohen_kappa_score(y_masculino.iloc[idx], y_feminino.iloc[idx])
    kappas_bootstrap.append(kappa_sample)

ic_lower = np.percentile(kappas_bootstrap, 2.5)
ic_upper = np.percentile(kappas_bootstrap, 97.5)

print(f"  IC 95%: [{ic_lower:.4f}, {ic_upper:.4f}]")

# Adicionar ao baseline
df_baseline_ic = pd.DataFrame({
    'metrica': ['Kappa_IC95_Lower', 'Kappa_IC95_Upper'],
    'valor': [ic_lower, ic_upper]
})
df_baseline = pd.concat([df_baseline, df_baseline_ic], ignore_index=True)

In [ ]:
# Salvar baseline
output_file = RESULTS_DIR / "metricas_consistencia_baseline.csv"
df_baseline.to_csv(output_file, index=False)
print(f"\n[OK] Baseline salvo: {output_file}")

In [ ]:
# Visualização: Distribuição de classes (Estilo Nature)
fig, axes = plt.subplots(1, 2, figsize=(7, 5))

# Cores semânticas para classes de sentimento
cores_classes = [COR_NEGATIVA, COR_NEUTRA, COR_POSITIVA]

for idx, grupo in enumerate(['masculino', 'feminino']):
    col = f'classificacao_majoritaria_{grupo}'
    dist = df[col].value_counts().sort_index()
    
    ax = axes[idx]
    bars = dist.plot(kind='bar', ax=ax, color=cores_classes, edgecolor='white', linewidth=1.5)
    ax.set_title(f'Grupo {grupo.capitalize()}', fontsize=12, fontweight='bold', color=NATURE_BLUE)
    ax.set_xlabel('Classe', fontsize=11)
    ax.set_ylabel('Frequência', fontsize=11)
    ax.set_xticklabels(['Negativa', 'Neutra', 'Positiva'], rotation=0)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    for container in ax.containers:
        ax.bar_label(container, fontsize=11, fontweight='bold', color=NATURE_BLUE)
    
    ax.set_ylim(0, dist.max() * 1.15)

plt.suptitle('Distribuição de Classes por Grupo de Anotadores', fontsize=14, fontweight='bold', color=NATURE_BLUE, y=1.02)
plt.tight_layout()
plt.savefig(GRAFICOS_DIR / 'empi_distribuicao_classes.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusão - Baseline
cm = confusion_matrix(y_masculino, y_feminino, labels=[-1, 0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Neg', 'Neu', 'Pos'],
            yticklabels=['Neg', 'Neu', 'Pos'],
            ax=ax, cbar_kws={'label': 'Frequência'},
            annot_kws={'fontsize': 16})

ax.set_title('Matriz de Confusão - Baseline (Anotadores)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Anotações Feminino', fontsize=12)
ax.set_ylabel('Anotações Masculino', fontsize=12)

plt.tight_layout()
plt.savefig(GRAFICOS_DIR / 'empi_matriz_confusao_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

## MÓDULO II: Treinamento Pareado dos Modelos

Treina modelos separadamente em cada grupo e avalia a concordância entre suas predições.

In [ ]:
# Preparar features TF-IDF
print("="*80)
print("MÓDULO II: TREINAMENTO PAREADO")
print("="*80)

stopwords_pt = [
    'a', 'o', 'e', 'é', 'de', 'da', 'do', 'em', 'um', 'uma', 'os', 'as', 'dos', 'das',
    'ao', 'à', 'às', 'no', 'na', 'nos', 'nas', 'por', 'para', 'com', 'sem', 'sob', 'sobre',
    'que', 'se', 'me', 'te', 'lhe', 'como', 'mas', 'ou', 'quando', 'onde', 'qual', 'quais',
    'ele', 'ela', 'eles', 'elas', 'eu', 'tu', 'nós', 'vós', 'você', 'vocês',
    'meu', 'minha', 'meus', 'minhas', 'seu', 'sua', 'seus', 'suas', 'nosso', 'nossa',
    'este', 'esta', 'esse', 'essa', 'aquele', 'aquela', 'isto', 'isso', 'aquilo',
    'foi', 'ser', 'ter', 'estar', 'fazer', 'ir', 'ver', 'dar', 'saber',
    'há', 'muito', 'mais', 'menos', 'bem', 'mal', 'já', 'ainda', 'só', 'também',
    'até', 'depois', 'antes', 'agora', 'então', 'aqui', 'ali', 'lá',
    'sim', 'não', 'nem', 'nunca', 'sempre', 'nada', 'tudo', 'todo', 'algum', 'nenhum'
]

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.90,
    strip_accents=None,
    stop_words=stopwords_pt,
    sublinear_tf=True,
    use_idf=True,
    norm='l2'
)

X = vectorizer.fit_transform(df['frase'])

print(f"\n[OK] Matriz TF-IDF: {X.shape}")
print(f"[OK] Vocabulário: {len(vectorizer.vocabulary_)} termos")

In [ ]:
# Configurar validação cruzada
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

resultados_modelos = []
predicoes_totais = []

print(f"\n[OK] StratifiedKFold: {N_FOLDS} folds")

In [ ]:
# Treinamento pareado com validação cruzada
print("\nIniciando treinamento pareado...\n")

inicio_treino = datetime.now()

for nome_modelo, modelo_template in modelos.items():
    print(f"  >>> {nome_modelo}")
    print(f"      " + "-"*40)
    
    metricas_folds = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_masculino), 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_masc_train, y_masc_test = y_masculino.iloc[train_idx], y_masculino.iloc[test_idx]
        y_fem_train, y_fem_test = y_feminino.iloc[train_idx], y_feminino.iloc[test_idx]
        
        # Treinar modelo masculino
        modelo_masc = clone(modelo_template)
        modelo_masc.fit(X_train, y_masc_train)
        pred_masc = modelo_masc.predict(X_test)
        
        # Treinar modelo feminino
        modelo_fem = clone(modelo_template)
        modelo_fem.fit(X_train, y_fem_train)
        pred_fem = modelo_fem.predict(X_test)
        
        # Métricas
        kappa_inter = cohen_kappa_score(pred_masc, pred_fem)
        concordancia_inter = np.mean(pred_masc == pred_fem)
        
        acc_masc = accuracy_score(y_masc_test, pred_masc)
        acc_fem = accuracy_score(y_fem_test, pred_fem)
        
        prec_masc, rec_masc, f1_masc, _ = precision_recall_fscore_support(
            y_masc_test, pred_masc, average='weighted', zero_division=0
        )
        prec_fem, rec_fem, f1_fem, _ = precision_recall_fscore_support(
            y_fem_test, pred_fem, average='weighted', zero_division=0
        )
        
        metricas_folds.append({
            'modelo': nome_modelo,
            'fold': fold,
            'kappa_inter_modelos': kappa_inter,
            'concordancia_inter_modelos': concordancia_inter,
            'acc_masculino': acc_masc,
            'acc_feminino': acc_fem,
            'precision_masculino': prec_masc,
            'precision_feminino': prec_fem,
            'recall_masculino': rec_masc,
            'recall_feminino': rec_fem,
            'f1_masculino': f1_masc,
            'f1_feminino': f1_fem
        })
        
        # Armazenar predições
        for i, idx in enumerate(test_idx):
            predicoes_totais.append({
                'modelo': nome_modelo,
                'fold': fold,
                'frase_id': int(idx),
                'frase': df.iloc[idx]['frase'][:100],
                'real_masculino': int(y_masc_test.iloc[i]),
                'real_feminino': int(y_fem_test.iloc[i]),
                'pred_masculino': int(pred_masc[i]),
                'pred_feminino': int(pred_fem[i]),
                'concordancia_real': int(y_masc_test.iloc[i] == y_fem_test.iloc[i]),
                'concordancia_pred': int(pred_masc[i] == pred_fem[i])
            })
        
        print(f"      Fold {fold}: κ={kappa_inter:.4f}, Acc_M={acc_masc:.4f}, Acc_F={acc_fem:.4f}")
    
    # Agregar
    df_folds = pd.DataFrame(metricas_folds)
    
    metricas_agregadas = {
        'modelo': nome_modelo,
        'kappa_mean': df_folds['kappa_inter_modelos'].mean(),
        'kappa_std': df_folds['kappa_inter_modelos'].std(),
        'concordancia_mean': df_folds['concordancia_inter_modelos'].mean(),
        'concordancia_std': df_folds['concordancia_inter_modelos'].std(),
        'acc_masculino_mean': df_folds['acc_masculino'].mean(),
        'acc_masculino_std': df_folds['acc_masculino'].std(),
        'acc_feminino_mean': df_folds['acc_feminino'].mean(),
        'acc_feminino_std': df_folds['acc_feminino'].std(),
        'f1_masculino_mean': df_folds['f1_masculino'].mean(),
        'f1_feminino_mean': df_folds['f1_feminino'].mean()
    }
    
    resultados_modelos.append(metricas_agregadas)
    
    print(f"      [OK] kappa_medio = {metricas_agregadas['kappa_mean']:.4f} +/- {metricas_agregadas['kappa_std']:.4f}\n")

duracao = datetime.now() - inicio_treino
print(f"[OK] Treinamento concluído em {duracao}")

In [ ]:
# Criar DataFrames de resultados
df_metricas_modelos = pd.DataFrame(resultados_modelos)
df_predicoes = pd.DataFrame(predicoes_totais)

print(f"\n[OK] Métricas: {df_metricas_modelos.shape}")
print(f"[OK] Predições: {df_predicoes.shape}")

df_metricas_modelos

In [ ]:
# Salvar resultados
df_metricas_modelos.to_csv(RESULTS_DIR / "metricas_modelos_pareados.csv", index=False)
df_predicoes.to_csv(RESULTS_DIR / "predicoes_completas.csv", index=False)

print("\n[OK] Arquivos salvos:")
print("  - metricas_modelos_pareados.csv")
print("  - predicoes_completas.csv")

## MÓDULO III: Avaliação de Divergência entre Modelos

Análise estatística das predições pareadas.

In [ ]:
# Análise estatística das predições
print("="*80)
print("MÓDULO III: DIVERGÊNCIA")
print("="*80)

analises = []

for modelo in modelos.keys():
    pred_modelo = df_predicoes[df_predicoes['modelo'] == modelo]
    
    contingency = pd.crosstab(pred_modelo['pred_masculino'], pred_modelo['pred_feminino'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    n = len(pred_modelo)
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0
    
    kappa_mean = df_metricas_modelos[df_metricas_modelos['modelo'] == modelo]['kappa_mean'].values[0]
    
    analises.append({
        'modelo': modelo,
        'kappa_modelos': kappa_mean,
        'chi2': chi2,
        'p_value': p_value,
        'cramers_v': cramers_v,
        'n_predicoes': n
    })
    
    print(f"\n  {modelo}:")
    print(f"    κ_modelos = {kappa_mean:.4f}")
    print(f"    χ² = {chi2:.2f} (p < {p_value:.2e})")
    print(f"    Cramér's V = {cramers_v:.4f}")

df_analises = pd.DataFrame(analises)
df_analises

In [ ]:
# Visualizar κ_inicial vs κ_modelos
fig, ax = plt.subplots(figsize=(7, 5))

x = np.arange(len(df_analises))
width = 0.5

bars = ax.bar(x, df_analises['kappa_modelos'], width,
              label='κ_modelos (Predições)', color=NATURE_BLUE_MED, alpha=0.9,
              edgecolor='white', linewidth=1.5)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
            f'{height:.4f}', ha='center', va='center', fontsize=10, 
            fontweight='bold', color='white')

ax.axhline(y=kappa_inicial, color=NATURE_BLUE, linestyle='--', alpha=0.7, linewidth=2.5,
           label=f'κ_inicial (Anotadores) = {kappa_inicial:.4f}')
ax.set_xlabel('Classificadores', fontsize=12, fontweight='bold')
ax.set_ylabel("Cohen's Kappa", fontsize=12, fontweight='bold')
ax.set_title('Concordância dos Classificadores x Concordância dos Anotadores', fontsize=14, fontweight='bold', color=NATURE_BLUE)
ax.set_xticks(x)
ax.set_xticklabels(df_analises['modelo'])
ax.legend(frameon=True, facecolor='white', edgecolor=NATURE_GRAY, fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, max(kappa_inicial, df_analises['kappa_modelos'].max()) * 1.15)

plt.tight_layout()
plt.savefig(GRAFICOS_DIR / 'empi_comparacao_kappas.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Matrizes de confusão dos modelos
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()

# Cmap azul acadêmico
cmap_nature = sns.light_palette(NATURE_BLUE, as_cmap=True)

for idx, modelo in enumerate(modelos.keys()):
    pred_modelo = df_predicoes[df_predicoes['modelo'] == modelo]
    
    cm = confusion_matrix(pred_modelo['pred_masculino'], pred_modelo['pred_feminino'],
                         labels=[-1, 0, 1])
    
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap_nature,
                xticklabels=['Neg', 'Neu', 'Pos'], 
                yticklabels=['Neg', 'Neu', 'Pos'],
                ax=ax, #cbar_kws={'label': 'Frequência'},
                annot_kws={'fontsize': 12, 'fontweight': 'bold'},
                linewidths=0.5, linecolor='white')
    
    kappa = df_metricas_modelos[df_metricas_modelos['modelo'] == modelo]['kappa_mean'].values[0]
    
    ax.set_title(f'{modelo} - κ = {kappa:.4f}', fontsize=11, fontweight='bold', color=NATURE_BLUE)
    ax.set_xlabel('Modelo Feminino', fontsize=10)
    ax.set_ylabel('Modelo Masculino', fontsize=10)

plt.suptitle('Matrizes de Contigência - Análise Pareada', fontsize=14, fontweight='bold', color=NATURE_BLUE)
plt.tight_layout()
plt.savefig(GRAFICOS_DIR / 'matrizes_confusao_modelos.png', dpi=300, bbox_inches='tight')
plt.show()

## MÓDULO IV: Comparação e Detecção de Amplificação

**Núcleo da MADA**: Calcula Δκ para detectar amplificação de viés.

**Critério de Severidade**: Baseado em **Landis & Koch (1977)**, pela queda de faixas de concordância.
- Referência: Landis JR, Koch GG (1977). "The measurement of observer agreement for categorical data". *Biometrics*, 33(1):159-174.

In [ ]:
# Calcular Δκ (Delta Kappa) com critério Landis & Koch (1977)
print("=" * 80)
print("MÓDULO IV: DETECÇÃO DE AMPLIFICAÇÃO (Critério: Landis & Koch, 1977)")
print("=" * 80)

# Faixas de concordância Kappa segundo Landis & Koch (1977)
LK_BANDS = [
    ('Poor', -1.0, 0.0),
    ('Slight', 0.0, 0.20),
    ('Fair', 0.20, 0.40),
    ('Moderate', 0.40, 0.60),
    ('Substantial', 0.60, 0.80),
    ('Almost Perfect', 0.80, 1.00),
]

LK_RANK = {name: i for i, (name, _, _) in enumerate(LK_BANDS)}


def obter_faixa_landis_koch(kappa):
    """Retorna a faixa de concordância segundo Landis & Koch (1977)."""
    if pd.isna(kappa):
        return np.nan
    for name, low, high in LK_BANDS:
        if kappa <= high:
            return name
    return 'Almost Perfect'


def classificar_severidade_por_queda_de_faixas(kappa_inicial, kappa_modelos):
    """Classifica severidade pela queda de faixas de concordância (Landis & Koch, 1977)."""
    faixa_inicial = obter_faixa_landis_koch(kappa_inicial)
    faixa_modelo = obter_faixa_landis_koch(kappa_modelos)

    faixas_perdidas = max(0, LK_RANK.get(faixa_inicial, 0) - LK_RANK.get(faixa_modelo, 0))

    SEVERIDADE = {0: "Ausente", 1: "Leve", 2: "Moderada"}
    severidade = SEVERIDADE.get(faixas_perdidas, "Severa")

    return severidade, faixas_perdidas, faixa_inicial, faixa_modelo


divergencias = []

for _, row in df_metricas_modelos.iterrows():
    delta_kappa = kappa_inicial - row['kappa_mean']
    severidade, faixas_perdidas, faixa_inicial, faixa_modelo = classificar_severidade_por_queda_de_faixas(
        kappa_inicial, row['kappa_mean']
    )

    divergencias.append({
        'modelo': row['modelo'],
        'kappa_inicial': kappa_inicial,
        'kappa_modelos': row['kappa_mean'],
        'delta_kappa': delta_kappa,
        'faixa_inicial_lk': faixa_inicial,
        'faixa_modelo_lk': faixa_modelo,
        'faixas_perdidas': faixas_perdidas,
        'severidade': severidade,
        'amplificacao_detectada': faixas_perdidas > 0
    })

    print(f"\n  {row['modelo']}:")
    print(f"    κ_inicial = {kappa_inicial:.4f} ({faixa_inicial})")
    print(f"    κ_modelos = {row['kappa_mean']:.4f} ({faixa_modelo})")
    print(f"    Δκ = {delta_kappa:+.4f}")
    print(f"    Faixas perdidas: {faixas_perdidas}")
    print(f"    Severidade: {severidade}")

df_divergencia = pd.DataFrame(divergencias)
df_divergencia


In [ ]:
# Salvar divergências
df_divergencia.to_csv(RESULTS_DIR / "divergencia_amplificacao.csv", index=False)
print("\n[OK] Salvo: divergencia_amplificacao.csv")

In [ ]:
# Visualizar Δκ com cores por severidade (critério Landis & Koch)
fig, ax = plt.subplots(figsize=(7.5, 5.5))

SEV_COLORS = {
    'Ausente': '#6B7280',
    'Leve': '#0284C7',
    'Moderada': '#F59E0B',
    'Severa': '#DC2626'
}

cores = [SEV_COLORS.get(sev, '#6B7280') for sev in df_divergencia['severidade']]

bars = ax.bar(df_divergencia['modelo'], df_divergencia['delta_kappa'],
              color=cores, alpha=0.85, edgecolor='white', linewidth=1.5)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1.2)

ax.set_xlabel('Classificadores', fontsize=12, fontweight='bold', color=NATURE_ACCENT)
ax.set_ylabel('Δκ (κ_inicial - κ_modelos)', fontsize=12, fontweight='bold', color=NATURE_ACCENT)
ax.set_title('Amplificação de Viés: Δκ por Classificador\n(Critério: Landis & Koch, 1977)',
             fontsize=13, fontweight='bold', color=NATURE_ACCENT, pad=15)
ax.grid(axis='y', alpha=0.3)

max_delta = df_divergencia['delta_kappa'].max()
margin = abs(max_delta) * 0.25
ax.set_ylim(0, max(max_delta + margin, 0.20))

for bar in bars:
    height = bar.get_height()
    if abs(height) > 0.03:
        ax.text(bar.get_x() + bar.get_width()/2., height / 2,
                f'{height:+.4f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='white')
    else:
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.015,
                f'{height:+.4f}', ha='center', va='bottom',
                fontsize=11, fontweight='bold', color='black')

legend_elements = [
    Patch(facecolor=SEV_COLORS['Ausente'], edgecolor='white', label='Ausente (0 faixas)'),
    Patch(facecolor=SEV_COLORS['Leve'], edgecolor='white', label='Leve (1 faixa)'),
    Patch(facecolor=SEV_COLORS['Moderada'], edgecolor='white', label='Moderada (2 faixas)'),
    Patch(facecolor=SEV_COLORS['Severa'], edgecolor='white', label='Severa (≥3 faixas)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8.5,
          title='Severidade', title_fontsize=9,
          frameon=True, facecolor='white', edgecolor=NATURE_GRAY,
          framealpha=0.95, shadow=False)

plt.tight_layout()
plt.savefig(GRAFICOS_DIR / 'empi_amplificacao_delta_kappa.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Bootstrap para significância estatística
print("\nCalculando significância estatística (bootstrap)...\n")

n_bootstrap = 1000
resultados_bootstrap = []

for modelo in modelos.keys():
    pred_modelo = df_predicoes[df_predicoes['modelo'] == modelo]
    
    deltas_bootstrap = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(pred_modelo), size=len(pred_modelo), replace=True)
        sample = pred_modelo.iloc[idx]
        
        kappa_sample = cohen_kappa_score(sample['pred_masculino'], sample['pred_feminino'])
        delta_sample = kappa_inicial - kappa_sample
        deltas_bootstrap.append(delta_sample)
    
    ci_lower = np.percentile(deltas_bootstrap, 2.5)
    ci_upper = np.percentile(deltas_bootstrap, 97.5)
    delta_mean = np.mean(deltas_bootstrap)
    
    significativo = ci_lower > 0 or ci_upper < 0
    direcao = "AMPLIFICA" if ci_lower > 0 else "REDUZ" if ci_upper < 0 else "NÃO SIGNIFICATIVO"
    
    resultados_bootstrap.append({
        'modelo': modelo,
        'delta_kappa': df_divergencia[df_divergencia['modelo'] == modelo]['delta_kappa'].values[0],
        'delta_bootstrap_mean': delta_mean,
        'ic_95_lower': ci_lower,
        'ic_95_upper': ci_upper,
        'significativo': significativo,
        'direcao': direcao
    })
    
    print(f"  {modelo}:")
    print(f"    Δκ observado: {resultados_bootstrap[-1]['delta_kappa']:+.4f}")
    print(f"    IC 95%: [{ci_lower:+.4f}, {ci_upper:+.4f}]")
    print(f"    Direcao: {direcao}")

df_bootstrap = pd.DataFrame(resultados_bootstrap)
df_bootstrap.to_csv(RESULTS_DIR / "significancia_delta_kappa.csv", index=False)

print("\n[OK] Salvo: significancia_delta_kappa.csv")

In [ ]:
# Gerar relatório textual
output_file = RESULTS_DIR / "relatorio_amplificacao_vies.txt"

BASELINE_FORMATS = {
    'Cohen_Kappa': '.4f', 'Concordancia': '.4f', 'Cramers_V': '.4f',
    'Kappa_IC95_Lower': '.4f', 'Kappa_IC95_Upper': '.4f',
    'Chi2': '.2f', 'p_value': '.2e', 'N_amostras': '.0f'
}

with open(output_file, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("RELATÓRIO DE AMPLIFICAÇÃO DE VIÉS - MADA FASE 2\n")
    f.write("=" * 80 + "\n\n")

    f.write(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
    f.write(f"Seed: {SEED}\n")
    f.write(f"K-Fold: {N_FOLDS}\n")
    f.write(f"N Amostras: {len(df)}\n\n")

    f.write("-" * 80 + "\n")
    f.write("BASELINE\n")
    f.write("-" * 80 + "\n\n")

    for _, row in df_baseline.iterrows():
        fmt = BASELINE_FORMATS.get(row['metrica'], '')
        f.write(f"{row['metrica']:25}: {row['valor']:{fmt}}\n")

    f.write("\n" + "-" * 80 + "\n")
    f.write("AMPLIFICAÇÃO\n")
    f.write("-" * 80 + "\n\n")

    f.write(f"κ_inicial: {kappa_inicial:.4f}\n\n")

    for _, row in df_divergencia.iterrows():
        f.write(f"{row['modelo']}:\n")
        f.write(f"  κ_modelos : {row['kappa_modelos']:.4f}\n")
        f.write(f"  Δκ        : {row['delta_kappa']:+.4f}\n")
        f.write(f"  Severidade: {row['severidade']}\n")
        f.write(f"  Amplifica?: {'SIM' if row['amplificacao_detectada'] else 'NÃO'}\n\n")

    f.write("=" * 80 + "\n")

    n_amplificados = df_divergencia['amplificacao_detectada'].sum()

    if n_amplificados == 0:
        f.write("[OK] Nenhum modelo amplificou viés significativamente.\n")
    else:
        f.write(f"[AVISO] {n_amplificados}/{len(df_divergencia)} modelos amplificaram viés.\n")

print(f"\n[OK] Relatório salvo: {output_file}")


## Resumo Final

In [ ]:
# Interpretação focada em amplificação (Critério: Landis & Koch, 1977)
print("="*80)
print("ANÁLISE ESPECÍFICA: AMPLIFICAÇÃO DE VIÉS (Landis & Koch, 1977)")
print("="*80)

print(f"\n{'MODELO':<10} | {'kappa_inicial':^14} | {'kappa_modelos':^14} | {'Delta_kappa':^12} | {'Faixas':^8} | {'SEVERIDADE':^15}")
print("-"*95)

for _, row in df_divergencia.iterrows():
    faixas = row['faixas_perdidas']
    severidade = row['severidade']
    
    print(f"{row['modelo']:<10} | {row['kappa_inicial']:^14.4f} | "
          f"{row['kappa_modelos']:^14.4f} | {row['delta_kappa']:^+12.4f} | "
          f"{faixas:^8} | {severidade:^15}")

print("-"*95)

delta_medio = df_divergencia['delta_kappa'].mean()
faixas_perdidas_media = df_divergencia['faixas_perdidas'].mean()

print(f"\nDelta kappa médio: {delta_medio:+.4f}")
print(f"   κ_inicial: {kappa_inicial:.4f}")
print(f"   κ_modelos (média): {df_divergencia['kappa_modelos'].mean():.4f}")
print(f"   Faixas perdidas (média): {faixas_perdidas_media:.2f}")

print("\n" + "="*80)
print("INTERPRETAÇÃO (Baseada em Landis & Koch, 1977):")
print("="*80)

n_amplificados = df_divergencia['amplificacao_detectada'].sum()
n_total = len(df_divergencia)

if n_amplificados > 0:
    print(f"\n[AVISO] AMPLIFICAÇÃO DETECTADA")
    print(f"   {n_amplificados}/{n_total} modelos apresentam perda de faixas de concordância")
    print(f"   Modelos concordam MENOS ({df_divergencia['kappa_modelos'].mean():.4f})")
    print(f"   que os anotadores ({kappa_inicial:.4f})")
    print(f"\n   Distribuição de severidade:")
    for sev in ['Leve', 'Moderada', 'Severa']:
        count = (df_divergencia['severidade'] == sev).sum()
        if count > 0:
            print(f"   - {sev}: {count} modelo(s)")
    print("\n   Conclusão: Modelos AMPLIFICAM as diferenças de perspectiva")
else:
    print(f"\n[OK] SEM AMPLIFICAÇÃO (Delta kappa medio = {delta_medio:+.4f})")
    print("   Conclusão: Modelos PRESERVAM o nível de divergência original")

print("\n" + "="*80)

In [ ]:
# =============================================================================
# RESUMO FINAL DE ARQUIVOS GERADOS
# =============================================================================

print("=" * 80)
print("FASE 2 — RESUMO DE ARQUIVOS GERADOS")
print("=" * 80)

# Verifica automaticamente quais arquivos foram gerados
for dirpath in [RESULTS_DIR, GRAFICOS_DIR]:
    for f in sorted(dirpath.iterdir()):
        if f.is_file():
            print(f"  [OK] {f.relative_to(DATA_DIR)}")

print(f"\n{'=' * 80}")
print("[OK] FASE 2 CONCLUÍDA COM SUCESSO")
print(f"{'=' * 80}")


In [ ]:
# Gerar arquivo Markdown com os principais insights estatísticos da Fase 2

insights_file = RESULTS_DIR / "insights_estatisticos_fase2.md"

# Extrair métricas do baseline via dict para acesso direto
baseline = df_baseline.set_index('metrica')['valor'].to_dict()

delta_medio = float(df_divergencia['delta_kappa'].mean())
kappa_modelos_medio = float(df_divergencia['kappa_modelos'].mean())

melhor_kappa = df_metricas_modelos.loc[df_metricas_modelos['kappa_mean'].idxmax()]
pior_kappa = df_metricas_modelos.loc[df_metricas_modelos['kappa_mean'].idxmin()]

maior_delta = df_divergencia.loc[df_divergencia['delta_kappa'].idxmax()]
menor_delta = df_divergencia.loc[df_divergencia['delta_kappa'].idxmin()]

dist_severidade = df_divergencia['severidade'].value_counts().to_dict()

n_significativos = int(df_bootstrap['significativo'].sum()) if 'df_bootstrap' in globals() else 0

linhas = [
    "# Principais Insights Estatísticos — MADA Fase 2",
    "",
    f"**Data de geração:** {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
    "",
    "## 1) Baseline entre anotadores",
    f"- **Cohen's Kappa inicial:** `{kappa_inicial:.4f}`",
    f"- **IC 95% do Kappa (bootstrap):** `[{baseline['Kappa_IC95_Lower']:.4f}, {baseline['Kappa_IC95_Upper']:.4f}]`",
    f"- **Concordância observada:** `{baseline['Concordancia']:.4f}` ({baseline['Concordancia']*100:.2f}%)",
    f"- **Cramér's V:** `{baseline['Cramers_V']:.4f}`",
    "",
    "> Interpretação: há concordância inicial relativamente alta entre os grupos de anotadores.",
    "",
    "## 2) Concordância entre modelos pareados",
    f"- **Média de κ_modelos (4 classificadores):** `{kappa_modelos_medio:.4f}`",
    f"- **Melhor κ_modelos:** `{melhor_kappa['modelo']}` com `{melhor_kappa['kappa_mean']:.4f}`",
    f"- **Pior κ_modelos:** `{pior_kappa['modelo']}` com `{pior_kappa['kappa_mean']:.4f}`",
    "",
    "## 3) Amplificação de viés (Δκ = κ_inicial - κ_modelos)",
    f"- **Δκ médio:** `{delta_medio:+.4f}`",
    f"- **Maior amplificação:** `{maior_delta['modelo']}` com `Δκ={maior_delta['delta_kappa']:+.4f}`",
    f"- **Menor amplificação:** `{menor_delta['modelo']}` com `Δκ={menor_delta['delta_kappa']:+.4f}`",
    f"- **Modelos com amplificação detectada:** `{int(n_amplificados)}/{int(n_total)}`",
    "",
    "### Severidade (Landis & Koch, 1977)",
]

for sev in ["Ausente", "Leve", "Moderada", "Severa"]:
    linhas.append(f"- **{sev}:** {dist_severidade.get(sev, 0)} modelo(s)")

linhas += [
    "",
    "## 4) Significância estatística (bootstrap de Δκ)",
    f"- **Modelos com Δκ estatisticamente significativo:** `{n_significativos}/{len(modelos)}`",
]

if 'df_bootstrap' in globals():
    for _, r in df_bootstrap.iterrows():
        linhas.append(
            f"  - {r['modelo']}: Δκ observado `{r['delta_kappa']:+.4f}`, "
            f"IC95% `[{r['ic_95_lower']:+.4f}, {r['ic_95_upper']:+.4f}]`, "
            f"direção: **{r['direcao']}**"
        )

linhas += [
    "",
    "## 5) Conclusão executiva",
    "- Os classificadores reproduzem padrões com **menor concordância inter-grupos** do que a observada entre anotadores humanos.",
    "- Houve **amplificação de viés em todos os modelos testados** nesta configuração experimental.",
    "- A queda de faixa de concordância foi predominantemente **leve** (1 faixa em Landis & Koch), mas consistente.",
    "",
    "---",
    f"Arquivo gerado automaticamente em: `{insights_file}`"
]

with open(insights_file, "w", encoding="utf-8") as f:
    f.write("\n".join(linhas))

print(f"[OK] Arquivo de insights salvo: {insights_file}")


# Principais Insights Estatísticos — MADA Fase 2

**Data de geração:** 08/04/2026 10:44:34

## 1) Baseline entre anotadores
- **Cohen's Kappa inicial:** `0.7664`
- **IC 95% do Kappa (bootstrap):** `[0.7363, 0.7954]`
- **Concordância observada:** `0.8453` (84.53%)
- **Cramér's V:** `0.7679`

> Interpretação: há concordância inicial relativamente alta entre os grupos de anotadores.

## 2) Concordância entre modelos pareados
- **Média de κ_modelos (4 classificadores):** `0.4877`
- **Melhor κ_modelos:** `LR` com `0.5120`
- **Pior κ_modelos:** `NB` com `0.4582`

## 3) Amplificação de viés (Δκ = κ_inicial - κ_modelos)
- **Δκ médio:** `+0.2787`
- **Maior amplificação:** `NB` com `Δκ=+0.3082`
- **Menor amplificação:** `LR` com `Δκ=+0.2544`
- **Modelos com amplificação detectada:** `4/4`

### Severidade (Landis & Koch, 1977)
- **Ausente:** 0 modelo(s)
- **Leve:** 4 modelo(s)
- **Moderada:** 0 modelo(s)
- **Severa:** 0 modelo(s)

## 4) Significância estatística (bootstrap de Δκ)
- **Modelos com Δκ estatisticamente significativo:** `0/4`

## 5) Conclusão executiva
- Os classificadores reproduzem padrões com **menor concordância inter-grupos** do que a observada entre anotadores humanos.
- Houve **amplificação de viés em todos os modelos testados** nesta configuração experimental.
- A queda de faixa de concordância foi predominantemente **leve** (1 faixa em Landis & Koch), mas consistente.

---
Arquivo gerado automaticamente em: `data\resultados_empiricos\insights_estatisticos_fase2.md`